<a href="https://colab.research.google.com/github/nataliamarinn/labo3-2026r/blob/main/src/AutoGluon/z336_SimpleStats.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/.drive')

In [ ]:
%%shell
mkdir -p "/content/.drive/My Drive/labo3"
mkdir -p /content/buckets
ln -sfn "/content/.drive/My Drive/labo3" /content/buckets/b1
mkdir -p ~/.kaggle
cp /content/buckets/b1/kaggle/kaggle.json ~/.kaggle
chmod 600 ~/.kaggle/kaggle.json
mkdir -p /content/buckets/b1/datasets /content/datasets

descargar() {
  d="/content/buckets/b1/datasets/"
  u="https://storage.googleapis.com/open-courses/austral2026-5da5/labo3/"
  if ! test -f "$d$1"; then wget "$u$1" -O "$d$1"; fi
  if ! test -f "/content/datasets/$1"; then cp "$d$1" "/content/datasets/$1"; fi
}
descargar sell-in.txt.gz
descargar product_id_apredecir201912.txt

In [ ]:
!pip install uv -q && uv pip install -q kaggle

In [ ]:
import os, itertools
import numpy as np
import polars as pl
from sklearn.linear_model import LinearRegression
import warnings
warnings.filterwarnings('ignore')

COMPETENCIA = 'labo-iii-2026-rosario'

dataset      = pl.read_csv('/content/.drive/My Drive/labo3/datasets/sell-in.txt.gz', separator='\t')
tb_ventas    = dataset.group_by('product_id','periodo').agg(pl.col('tn').sum()).sort(['product_id','periodo'])
tb_apredecir = pl.read_csv('/content/.drive/My Drive/labo3/datasets/product_id_apredecir201912.txt', separator='\t')
tb_ventas    = tb_ventas.join(tb_apredecir, on='product_id', how='inner').sort(['product_id','periodo'])
productos    = tb_apredecir['product_id'].to_list()
print(f'{len(productos)} productos')

In [ ]:
# ── predictores ──────────────────────────────────────────────────────────────

def mediana_medianas(serie, ventana, bloque=3):
    """Divide los últimos `ventana` meses en bloques de `bloque`, mediana de cada bloque,
    después mediana de esas medianas. Robusta a outliers y cambios de nivel."""
    s = serie[-min(ventana, len(serie)):]
    bloques = [s[i:i+bloque] for i in range(0, len(s), bloque) if len(s[i:i+bloque]) > 0]
    return max(float(np.median([np.median(b) for b in bloques])), 0.0)


def weighted_median(serie, ventana):
    """Mediana ponderada: pesos lineales crecientes (más reciente = más peso)."""
    s = serie[-min(ventana, len(serie)):]
    n = len(s)
    pesos = np.arange(1, n + 1, dtype=float)
    pesos /= pesos.sum()
    # mediana ponderada: sort + cumsum de pesos hasta 0.5
    orden = np.argsort(s)
    s_ord = s[orden]
    p_ord = pesos[orden]
    cum   = np.cumsum(p_ord)
    idx   = np.searchsorted(cum, 0.5)
    return max(float(s_ord[min(idx, len(s_ord)-1)]), 0.0)


def trimmed_mean(serie, ventana, trim=0.1):
    """Media recortando `trim` fracción de los extremos (sup e inf)."""
    s = serie[-min(ventana, len(serie)):]
    n = len(s)
    k = max(1, int(n * trim))
    s_sorted = np.sort(s)
    trimmed  = s_sorted[k:-k] if 2*k < n else s_sorted
    return max(float(trimmed.mean()), 0.0)


def reg_lineal(serie, ventana, horizonte=2):
    """OLS sobre últimos `ventana` meses, extrapola `horizonte` pasos."""
    w = min(ventana, len(serie))
    if w < 2:
        return max(float(serie.mean()), 0.0)
    y = serie[-w:]
    x = np.arange(w).reshape(-1, 1)
    pred = float(LinearRegression().fit(x, y).predict([[w - 1 + horizonte]])[0])
    return max(pred, 0.0)


print('Funciones OK')

In [ ]:
# ── variantes a probar ────────────────────────────────────────────────────────
# cada entrada: (nombre, función, ventanas a barrer)

ventanas = [3, 6, 9, 12, 18, 24]

variantes = []
for v in ventanas:
    variantes += [
        (f'medmed_{v}m',   lambda s, v=v: mediana_medianas(s, v)),
        (f'wmedian_{v}m',  lambda s, v=v: weighted_median(s, v)),
        (f'trimmed_{v}m',  lambda s, v=v: trimmed_mean(s, v)),
        (f'reg_{v}m',      lambda s, v=v: reg_lineal(s, v)),
    ]

print(f'{len(variantes)} variantes a submitear')

In [ ]:
# ── precalcular series una sola vez ──────────────────────────────────────────
series_dict = {}
for pid in productos:
    series_dict[pid] = (
        tb_ventas.filter(pl.col('product_id') == pid)
        .sort('periodo')['tn'].to_numpy().astype(float)
    )

def kaggle_submit(competencia, archivo, mensaje):
    os.system(f'kaggle competitions submit -c {competencia} -f {archivo} -m "{mensaje}"')

# ── loop de submits ───────────────────────────────────────────────────────────
for nombre, fn in variantes:
    preds = [{'product_id': pid, 'tn': fn(series_dict[pid])} for pid in productos]
    tb_final = pl.DataFrame(preds)
    archivo  = f"{nombre}.csv"
    tb_final.write_csv(archivo)
    kaggle_submit(COMPETENCIA, archivo, nombre)
    print(f'submitted: {nombre}')